In [1]:
from deutschland import smard
import time
import requests
from deutschland import smard
from deutschland.smard.api import default_api
from deutschland.smard.model.indices import Indices
from deutschland.smard.model.time_series import TimeSeries
from datetime import datetime
import pandas as pd
from io import BytesIO
url = 'https://www.smard.de/nip-download-manager/nip/download/market-data'

In [2]:
start = datetime(2024, 10, 25, 00, 00, 00)
end =  datetime(2024, 10, 26, 0, 0, 0)


In [ ]:
payload = {"request_form":[{"format":"CSV","moduleIds":[1004067,1004068,1001225],"region":"DE","timestamp_from":int(start.timestamp())*1000,"timestamp_to":int(end.timestamp())*1000,"type":"discrete","language":"de"}]}
energy = requests.post(url, json = payload)
df_energy = pd.read_csv(BytesIO(energy.content), sep=';')

In [ ]:
df_energy.columns=['Time_from','Time_to','Windoffshore','Windonshore','Solar']

df_energy['Windoffshore'] = df_energy['Windoffshore'].str.replace('.', '')
df_energy['Windoffshore'] = df_energy['Windoffshore'].str.replace(',', '.')
df_energy['Windoffshore'] = df_energy['Windoffshore'].str.replace('-', '0')
df_energy['Windoffshore'] = df_energy['Windoffshore'].astype(float)

df_energy['Windonshore'] = df_energy['Windonshore'].str.replace('.', '')
df_energy['Windonshore'] = df_energy['Windonshore'].str.replace(',', '.')
df_energy['Windonshore'] = df_energy['Windonshore'].str.replace('-', '0')
df_energy['Windonshore'] = df_energy['Windonshore'].astype(float)

df_energy['Solar'] = df_energy['Solar'].str.replace('.', '')
df_energy['Solar'] = df_energy['Solar'].str.replace(',', '.')
df_energy['Solar'] = df_energy['Solar'].str.replace('-', '0')
df_energy['Solar'] = df_energy['Solar'].astype(float)
#df_energy

In [4]:
payload = {"request_form":[{"format":"CSV","moduleIds":[8004169],"region":"DE","timestamp_from":int(start.timestamp())*1000,"timestamp_to":int(end.timestamp())*1000,"type":"discrete","language":"de"}]}
price = requests.post(url, json = payload)
df_price = pd.read_csv(BytesIO(price.content), sep=';')
df_price.columns=['Time_from','Time_to','Market_price']

df_price['Market_price'] = df_price['Market_price'].str.replace('.', '')
df_price['Market_price'] = df_price['Market_price'].str.replace(',', '.')
df_price['Market_price'] = df_price['Market_price'].str.replace('-', '0')
df_price['Market_price'] = df_price['Market_price'].astype(float)

#df_price

In [5]:
payload = {"request_form":[{"format":"CSV","moduleIds":[5000410],"region":"DE","timestamp_from":int(start.timestamp())*1000,"timestamp_to":int(end.timestamp())*1000,"type":"discrete","language":"de"}]}
load = requests.post(url, json = payload)
df_load = pd.read_csv(BytesIO(load.content), sep=';')
df_load.columns=['Time_from','Time_to','Load']
#df_load 
df_load['Load'] = df_load['Load'].str.replace('.', '')
df_load['Load'] = df_load['Load'].str.replace(',', '.')
df_load['Load'] = df_load['Load'].str.replace('-', '0')
df_load['Load'] = df_load['Load'].astype(float)

#df_load

In [ ]:
df_inter = pd.merge(df_energy, df_load, on=['Time_from','Time_to'])
df = pd.merge(df_inter, df_price, on=['Time_from'], how = 'outer')
df = df.drop(columns=['Time_to_x', 'Time_to_y'])
#df.columns=['Time','Windoffshore','Windonshore','Solar','Load','Market_price']

df['Wind'] = df['Windoffshore'] + df['Windonshore']

df = df.drop(columns=['Windoffshore', 'Windonshore'])
df['Market_price']= df['Market_price'].ffill()
df.set_index('Time_from')
df['Time_from'] = pd.to_datetime(df['Time_from'],dayfirst=True)
df_reordered = df.loc[:, ['Time_from','Market_price','Load','Wind','Solar']] 
df_reordered.to_csv('data/SMARD_data.csv', sep='\t',index=False)

Time_from       datetime64[ns]
Market_price           float64
Load                   float64
Wind                   float64
Solar                  float64
dtype: object
